In [ ]:
# Load all cleaned country datasets
data_dir = 'data'
countries = ['ethiopia', 'kenya', 'sudan', 'tanzania', 'nigeria']
dfs = []
for country in countries:
    path = os.path.join(data_dir, f'{country}_clean.csv')
    df = pd.read_csv(path)
    df['Country'] = country.capitalize()
    dfs.append(df)
all_df = pd.concat(dfs, ignore_index=True)
all_df['date'] = pd.to_datetime(all_df['date'])
all_df['Month'] = all_df['date'].dt.month
all_df.head()

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# Cross-Country Climate Comparison

This notebook synthesizes cleaned climate data from Ethiopia, Kenya, Sudan, Tanzania, and Nigeria to compare temperature trends, precipitation variability, and extreme event frequency. It produces a data-driven vulnerability ranking to inform Ethiopia's COP32 position paper.

# Cross-Country Climate Comparison\n
Task 3 notebook for Ethiopia, Kenya, Sudan, Tanzania, and Nigeria.

In [ ]:
import numpy as np\n
import pandas as pd\n
import matplotlib.pyplot as plt\n
import seaborn as sns\n
from scipy.stats import f_oneway, kruskal

In [ ]:
countries = ['ethiopia', 'kenya', 'sudan', 'tanzania', 'nigeria']\n
frames = []\n
for c in countries:\n
    temp = pd.read_csv(f'../data/{c}_clean.csv')\n
    if 'DATE' in temp.columns:\n
        temp['DATE'] = pd.to_datetime(temp['DATE'])\n
    else:\n
        temp['DATE'] = pd.to_datetime(temp['YEAR'] * 1000 + temp['DOY'], format='%Y%j')\n
    temp['Country'] = c.title()\n
    frames.append(temp)\n
df = pd.concat(frames, ignore_index=True)\n
df['Year'] = df['DATE'].dt.year

In [ ]:
monthly_t2m = df.groupby(['Country', pd.Grouper(key='DATE', freq='M')])['T2M'].mean().reset_index()\n
plt.figure(figsize=(13, 5))\n
sns.lineplot(data=monthly_t2m, x='DATE', y='T2M', hue='Country')\n
plt.title('Monthly Average T2M by Country')\n
plt.show()

In [ ]:
t2m_summary = df.groupby('Country')['T2M'].agg(['mean', 'median', 'std']).sort_values('mean', ascending=False)\n
prect_summary = df.groupby('Country')['PRECTOTCORR'].agg(['mean', 'median', 'std']).sort_values('mean', ascending=False)\n
display(t2m_summary)\n
display(prect_summary)

In [ ]:
plt.figure(figsize=(10, 5))\n
sns.boxplot(data=df, x='Country', y='PRECTOTCORR')\n
plt.title('PRECTOTCORR Variability by Country')\n
plt.show()

In [ ]:
extreme_heat = (df.assign(extreme_heat=df['T2M_MAX'] > 35)\n
                .groupby(['Country', 'Year'])['extreme_heat'].sum()\n
                .reset_index())\n
plt.figure(figsize=(12, 5))\n
sns.barplot(data=extreme_heat, x='Year', y='extreme_heat', hue='Country')\n
plt.title('Extreme Heat Days per Year (T2M_MAX > 35°C)')\n
plt.xticks(rotation=45)\n
plt.show()

In [ ]:
def consecutive_dry_days(country_df, threshold=1.0):\n
    country_df = country_df.sort_values('DATE').copy()\n
    dry = (country_df['PRECTOTCORR'] < threshold).astype(int).values\n
    max_run = 0\n
    run = 0\n
    for v in dry:\n
        if v == 1:\n
            run += 1\n
            max_run = max(max_run, run)\n
        else:\n
            run = 0\n
    return max_run\n
\n
dry_rows = []\n
for (country, year), g in df.groupby(['Country', 'Year']):\n
    dry_rows.append({'Country': country, 'Year': year, 'max_consecutive_dry_days': consecutive_dry_days(g)})\n
dry_df = pd.DataFrame(dry_rows)\n
plt.figure(figsize=(12, 5))\n
sns.barplot(data=dry_df, x='Year', y='max_consecutive_dry_days', hue='Country')\n
plt.title('Max Consecutive Dry Days per Year (PRECTOTCORR < 1 mm)')\n
plt.xticks(rotation=45)\n
plt.show()

In [ ]:
samples = [grp['T2M'].dropna().values for _, grp in df.groupby('Country')]\n
anova_stat, anova_p = f_oneway(*samples)\n
kruskal_stat, kruskal_p = kruskal(*samples)\n
print('ANOVA p-value:', anova_p)\n
print('Kruskal-Wallis p-value:', kruskal_p)

In [ ]:
ranking = (df.groupby('Country')\n
           .agg(mean_t2m=('T2M', 'mean'),\n
                temp_std=('T2M', 'std'),\n
                mean_prec=('PRECTOTCORR', 'mean'),\n
                prec_std=('PRECTOTCORR', 'std'))\n
           .reset_index())\n
ranking['vulnerability_score'] = ranking['mean_t2m'].rank(ascending=False) + ranking['prec_std'].rank(ascending=False)\n
ranking = ranking.sort_values('vulnerability_score', ascending=False)\n
display(ranking)